## Exploración de estructura entre eras

Objetivo: Confirmar qué columnas cambian entre temporadas viejas y nuevas antes de escribir cualquier lógica de limpieza.

In [21]:
import pandas as pd

season_9394 = pd.read_csv('../data/raw/results/season-9394.csv')
season_0405 = pd.read_csv('../data/raw/results/season-0405.csv')
season_0506 = pd.read_csv('../data/raw/results/season-0506.csv')

for nombre, df in [('93-94', season_9394), ('04-05', season_0405), ('05-06', season_0506)]:
    print(nombre, df.shape)
    print(df.columns.tolist())
    print()

93-94 (380, 22)
['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']

04-05 (380, 22)
['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']

05-06 (380, 22)
['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']



In [22]:
print('93-94 nulos en HS:', season_9394['HS'].isna().sum(), 'de', len(season_9394))
print('04-05 nulos en HS:', season_0405['HS'].isna().sum(), 'de', len(season_0405))
print('05-06 nulos en HS:', season_0506['HS'].isna().sum(), 'de', len(season_0506))

93-94 nulos en HS: 380 de 380
04-05 nulos en HS: 380 de 380
05-06 nulos en HS: 0 de 380


### Hallazgo: disponibilidad de estadísticas de juego por era

Las estadísticas de juego (tiros, tiros a puerta, córners, tarjetas, faltas: columnas `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR`) solo están disponibles desde la temporada **2005/06** en adelante. En las temporadas 1993/94 a 2004/05, esas columnas existen en el CSV pero están 100% vacías.

**Implicación para el análisis:**
- Cualquier métrica de rendimiento basada en goles, resultado o puntos (`FTHG`, `FTAG`, `FTR`) cubre el rango completo, 1993/94 en adelante.
- Cualquier análisis que dependa de estadísticas de juego (posesión indirecta vía tiros, disciplina vía tarjetas, etc.) queda limitado a partir de 2005/06.

Esto no afecta la métrica principal de "temporada buena vs mala" definida más adelante (puntos, posición, diferencia de gol), que solo depende de goles y resultado.

In [23]:
equipo = 'Barcelona'

barca_9394 = season_9394[(season_9394['HomeTeam'] == equipo) | (season_9394['AwayTeam'] == equipo)]
barca_0405 = season_0405[(season_0405['HomeTeam'] == equipo) | (season_0405['AwayTeam'] == equipo)]
barca_0506 = season_0506[(season_0506['HomeTeam'] == equipo) | (season_0506['AwayTeam'] == equipo)]

for nombre, df in [('93-94', barca_9394), ('04-05', barca_0405), ('05-06', barca_0506)]:
    print(nombre, df.shape)

93-94 (38, 22)
04-05 (38, 22)
05-06 (38, 22)


### Reconstrucción de tabla de posiciones por temporada

Los CSVs de football-data.co.uk traen resultados partido a partido de toda la liga, no la tabla de posiciones ya calculada. Para saber en qué posición terminó el Barça cada temporada, hay que reconstruir la tabla completa de los 20 equipos (agregando puntos, victorias, empates, derrotas, goles a favor/contra y diferencia de gol) a partir de esos resultados.

**Decisión de diseño:** los puntos se calculan con la regla vigente en cada temporada (2 puntos por victoria antes de 1995/96, 3 puntos después), no normalizados. Esto respeta la clasificación real de cada año. Cualquier comparación de puntos entre una temporada de la era de 2 y una de la era de 3 se normaliza puntualmente en el momento de esa comparación específica, no en el dato base.

**Criterio de desempate:** diferencia de gol como único criterio secundario (los criterios oficiales de La Liga incluyen más factores como resultados entre los equipos empatados). Esto solo se valida manualmente contra la clasificación real cuando el desempate afecta una posición relevante (título, zona de Champions/Europa, descenso).

In [24]:
def construir_tabla_liga(df, temporada):
    puntos_victoria = 2 if temporada in ['9394', '9495'] else 3

    local = df[['HomeTeam', 'FTHG', 'FTAG']].copy()
    local.columns = ['equipo', 'goles_favor', 'goles_contra']

    visitante = df[['AwayTeam', 'FTAG', 'FTHG']].copy()
    visitante.columns = ['equipo', 'goles_favor', 'goles_contra']

    partidos = pd.concat([local, visitante], ignore_index=True)

    partidos['resultado'] = 'empate'
    partidos.loc[partidos['goles_favor'] > partidos['goles_contra'], 'resultado'] = 'victoria'
    partidos.loc[partidos['goles_favor'] < partidos['goles_contra'], 'resultado'] = 'derrota'

    partidos['puntos'] = partidos['resultado'].map({
        'victoria': puntos_victoria,
        'empate': 1,
        'derrota': 0
    })

    tabla = partidos.groupby('equipo').agg(
        partidos_jugados=('resultado', 'count'),
        victorias=('resultado', lambda x: (x == 'victoria').sum()),
        empates=('resultado', lambda x: (x == 'empate').sum()),
        derrotas=('resultado', lambda x: (x == 'derrota').sum()),
        goles_favor=('goles_favor', 'sum'),
        goles_contra=('goles_contra', 'sum'),
        puntos=('puntos', 'sum')
    ).reset_index()

    tabla['diferencia_gol'] = tabla['goles_favor'] - tabla['goles_contra']
    tabla = tabla.sort_values(['puntos', 'diferencia_gol'], ascending=False).reset_index(drop=True)
    tabla['posicion'] = tabla.index + 1

    return tabla

In [25]:
tabla_9394 = construir_tabla_liga(season_9394, '9394')
tabla_0405 = construir_tabla_liga(season_0405, '0405')
tabla_0506 = construir_tabla_liga(season_0506, '0506')

for nombre, tabla in [('93-94', tabla_9394), ('04-05', tabla_0405), ('05-06', tabla_0506)]:
    fila_barca = tabla[tabla['equipo'] == 'Barcelona']
    print(nombre, '- Posición:', fila_barca['posicion'].values[0], '- Puntos:', fila_barca['puntos'].values[0])

93-94 - Posición: 1 - Puntos: 56
04-05 - Posición: 1 - Puntos: 84
05-06 - Posición: 1 - Puntos: 82


### Validación contra la clasificación real

Se contrastaron las tres temporadas de muestra contra la clasificación histórica real de La Liga:

| Temporada | Posición calculada | Puntos calculados | Resultado real |
|---|---|---|---|
| 1993/94 | 1 | 56 | Campeón (confirmado, 4to título consecutivo bajo Cruyff) |
| 2004/05 | 1 | 84 | Campeón (confirmado, bajo Rijkaard) |
| 2005/06 | 1 | 82 | Campeón (confirmado, bicampeón bajo Rijkaard) |

Las tres posiciones coinciden con la realidad, incluyendo una temporada anterior al cambio de regla de puntos (93-94, con 2 puntos por victoria) y dos posteriores. Esto valida tanto la lógica de agregación (`groupby` por equipo sumando ambas perspectivas, local y visitante) como el manejo del cambio de puntuación por era.

Con esta validación, la función queda lista para aplicarse al conjunto completo de temporadas (1993/94 en adelante) sin necesidad de revisar cada una manualmente, salvo casos puntuales donde el resultado se vea inconsistente (por ejemplo, un año con menos de 20 equipos en la tabla, señal de nombre de equipo inconsistente o temporada con formato distinto).

In [26]:
import pandas as pd
from pathlib import Path

carpeta_resultados = Path('../data/raw/results')
archivos = sorted(carpeta_resultados.glob('season-*.csv'))

temporadas = {}
for archivo in archivos:
    codigo = archivo.stem.replace('season-', '')
    df = pd.read_csv(archivo)
    temporadas[codigo] = df
    print(codigo, df.shape)

0001 (380, 22)
0102 (380, 22)
0203 (380, 22)
0304 (380, 22)
0405 (380, 22)
0506 (380, 22)
0607 (380, 22)
0708 (380, 22)
0809 (380, 22)
0910 (380, 22)
1011 (380, 22)
1112 (380, 22)
1213 (380, 22)
1314 (380, 22)
1415 (380, 22)
1516 (380, 22)
1617 (380, 22)
1718 (380, 22)
1819 (380, 22)
1920 (380, 22)
2021 (380, 22)
2122 (380, 22)
2223 (380, 22)
2324 (380, 22)
2425 (380, 22)
2526 (380, 22)
9394 (380, 22)
9495 (380, 22)
9596 (462, 22)
9697 (462, 22)
9798 (380, 22)
9899 (380, 22)
9900 (380, 22)


### Carga masiva de las 33 temporadas (1993/94 a 2025/26)

Se automatizó la descarga de todas las temporadas restantes con un script (`src/descargar_temporadas.py`) que genera los códigos de temporada por rango de años, descarga cada CSV desde datahub.io, y omite los que ya existen localmente (idempotente, se puede volver a correr sin duplicar trabajo).

Con las 33 temporadas descargadas, se cargaron todas en memoria (`pd.read_csv` por archivo) y se validó el número de filas de cada una antes de avanzar:

- **31 temporadas con 380 partidos**: formato estándar de 20 equipos, todos contra todos ida y vuelta (20 × 19 = 380).
- **2 temporadas con 462 partidos** (1995/96 y 1996/97): la llamada "liga de los 22", un formato excepcional y transitorio con 22 equipos en vez de 20 (22 × 21 = 462), antes de volver al formato estándar en 1997/98.

Ningún archivo cargó con una estructura inesperada fuera de estos dos casos ya documentados, lo que confirma que las 33 temporadas están listas para procesarse de forma consistente.

**Nota pendiente:** el diccionario de temporadas quedó ordenado alfabéticamente por el código de string (`sorted()`), no cronológicamente. Se resuelve más adelante agregando una columna de año de inicio real, necesaria para cualquier visualización de serie de tiempo.

In [27]:
equipo = 'Barcelona'
partidos_barca = {}

for codigo, df in temporadas.items():
    filtro = (df['HomeTeam'] == equipo) | (df['AwayTeam'] == equipo)
    barca = df[filtro].copy()
    partidos_barca[codigo] = barca
    print(codigo, barca.shape)

0001 (38, 22)
0102 (38, 22)
0203 (38, 22)
0304 (38, 22)
0405 (38, 22)
0506 (38, 22)
0607 (38, 22)
0708 (38, 22)
0809 (38, 22)
0910 (38, 22)
1011 (38, 22)
1112 (38, 22)
1213 (38, 22)
1314 (38, 22)
1415 (38, 22)
1516 (38, 22)
1617 (38, 22)
1718 (38, 22)
1819 (38, 22)
1920 (38, 22)
2021 (38, 22)
2122 (38, 22)
2223 (38, 22)
2324 (38, 22)
2425 (38, 22)
2526 (38, 22)
9394 (38, 22)
9495 (38, 22)
9596 (42, 22)
9697 (42, 22)
9798 (38, 22)
9899 (38, 22)
9900 (38, 22)


In [28]:
def goles_barca(row):
    return row['FTHG'] if row['HomeTeam'] == equipo else row['FTAG']

def goles_rival(row):
    return row['FTAG'] if row['HomeTeam'] == equipo else row['FTHG']

for codigo, df in partidos_barca.items():
    df['goles_barca'] = df.apply(goles_barca, axis=1)
    df['goles_rival'] = df.apply(goles_rival, axis=1)
    df['temporada'] = codigo

In [29]:
partidos_barca_completo = pd.concat(partidos_barca.values(), ignore_index=True)
partidos_barca_completo[['temporada', 'Date', 'HomeTeam', 'AwayTeam', 'goles_barca', 'goles_rival']].head(10)

,temporada,Date,HomeTeam,AwayTeam,goles_barca,goles_rival
0,0001,2000-09-09,Barcelona,Malaga,2,1
1,0001,2000-09-16,Ath Bilbao,Barcelona,1,3
2,0001,2000-09-23,Barcelona,Santander,3,1
3,0001,2000-10-01,La Coruna,Barcelona,0,2
4,0001,2000-10-14,Sociedad,Barcelona,6,0
5,0001,2000-10-21,Barcelona,Real Madrid,2,0
6,0001,2000-10-28,Mallorca,Barcelona,0,2
7,0001,2000-11-01,Barcelona,Numancia,1,1
8,0001,2000-11-04,Las Palmas,Barcelona,1,0
9,0001,2000-11-12,Barcelona,Villarreal,1,2


In [30]:
print(len(partidos_barca_completo))

1262


### Cálculo de goles del Barça vs goles del rival, independiente de local/visitante

`FTHG` y `FTAG` en el CSV original representan goles del equipo local y visitante respectivamente, no goles del Barça de forma directa. Se calcularon dos columnas nuevas, `goles_barca` y `goles_rival`, que resuelven esto según si el Barça jugó como local o visitante en cada partido, aplicadas sobre las 33 temporadas completas (no solo las 3 de muestra iniciales).

Se agregó también una columna `temporada` a cada partido, necesaria para conservar la trazabilidad al unir las 33 tablas individuales en un solo DataFrame (`partidos_barca_completo`) con `pd.concat()`.

**Validación de estructura:** el conteo de filas antes del filtro (38 partidos por temporada estándar, 42 en las dos temporadas de 22 equipos) se confirmó consistente en las 33 temporadas sin excepciones, y el nombre "Barcelona" no requirió ningún ajuste de limpieza en este dataset específico. El total de partidos filtrados coincide con lo esperado: 1,262 (38 × 31 + 42 × 2).

**Validación contra resultados reales:** se contrastaron partidos puntuales de `partidos_barca_completo` contra fuentes externas. El resultado Barcelona 2-0 Real Madrid del 21/10/2000 y la goleada Real Sociedad 0-6 Barcelona del 14/10/2000 (la mayor victoria como visitante de esa temporada) coinciden exactamente, incluyendo el caso de partido como visitante, que es donde más fácilmente se invertirían por error los goles del Barça y del rival.

In [31]:
filas_barca = []

for codigo, df in temporadas.items():
    tabla = construir_tabla_liga(df, codigo)
    fila = tabla[tabla['equipo'] == 'Barcelona'].copy()
    fila['temporada'] = codigo
    filas_barca.append(fila)

rendimiento_barca = pd.concat(filas_barca, ignore_index=True)
rendimiento_barca = rendimiento_barca[['temporada', 'posicion', 'puntos', 'partidos_jugados',
                                         'victorias', 'empates', 'derrotas',
                                         'goles_favor', 'goles_contra', 'diferencia_gol']]
rendimiento_barca.head(10)

,temporada,posicion,puntos,partidos_jugados,victorias,empates,derrotas,goles_favor,goles_contra,diferencia_gol
0,0001,4,63,38,17,12,9,80,57,23
1,0102,4,64,38,18,10,10,65,37,28
2,0203,6,56,38,15,11,12,63,47,16
3,0304,2,72,38,21,9,8,63,39,24
4,0405,1,84,38,25,9,4,73,29,44
5,0506,1,82,38,25,7,6,80,35,45
6,0607,1,76,38,22,10,6,78,33,45
7,0708,3,67,38,19,10,9,76,43,33
8,0809,1,87,38,27,6,5,105,35,70
9,0910,1,99,38,31,6,1,98,24,74


In [32]:
def anio_inicio(codigo):
    anio = int(codigo[:2])
    return 1900 + anio if anio >= 90 else 2000 + anio

rendimiento_barca['anio_inicio'] = rendimiento_barca['temporada'].apply(anio_inicio)
rendimiento_barca = rendimiento_barca.sort_values('anio_inicio').reset_index(drop=True)
rendimiento_barca.head(10)

,temporada,posicion,puntos,partidos_jugados,victorias,empates,derrotas,goles_favor,goles_contra,diferencia_gol,anio_inicio
0,9394,1,56,38,25,6,7,91,42,49,1993
1,9495,4,46,38,18,10,10,60,45,15,1994
2,9596,3,80,42,22,14,6,72,39,33,1995
3,9697,2,90,42,28,6,8,102,48,54,1996
4,9798,1,74,38,23,5,10,78,56,22,1997
5,9899,1,79,38,24,7,7,87,43,44,1998
6,9900,2,64,38,19,7,12,70,46,24,1999
7,0001,4,63,38,17,12,9,80,57,23,2000
8,0102,4,64,38,18,10,10,65,37,28,2001
9,0203,6,56,38,15,11,12,63,47,16,2002


### Tabla de rendimiento del Barça por temporada (1993/94 a 2025/26)

Se aplicó `construir_tabla_liga` a las 33 temporadas cargadas, extrayendo únicamente la fila del Barça de cada tabla de posiciones y apilándolas en un solo DataFrame (`rendimiento_barca`), con una columna por temporada: posición, puntos, partidos jugados, victorias, empates, derrotas, goles a favor, goles en contra y diferencia de gol.

Se agregó una columna `anio_inicio` calculada a partir del código de temporada (por ejemplo, `9394` → 1993, `0506` → 2005), necesaria para ordenar cronológicamente y para cualquier visualización de serie de tiempo más adelante. El código de dos dígitos requiere una regla explícita de siglo: valores desde 90 en adelante pertenecen al siglo XX, valores menores al XXI.

**Validación contra hechos históricos conocidos:** se contrastaron dos temporadas específicas del DataFrame resultante contra fuentes externas.

- **2009/10:** el DataFrame da 31 victorias, 6 empates, 1 derrota, 99 puntos. Esto coincide exactamente con el récord histórico de puntos de La Liga en su momento, bajo Pep Guardiola, con la única derrota de la temporada ante el Atlético de Madrid.
- **1993/94, 2004/05, 2005/06:** posición 1 (campeón) confirmada en las tres, validado previamente contra la clasificación histórica real.

Con estas validaciones independientes en distintos puntos del rango temporal (incluyendo antes y después del cambio de puntuación de 1995/96), el DataFrame `rendimiento_barca` queda confiable como base para el resto del análisis.

### Persistencia de resultados intermedios

Se guardan en `data/processed/` los dos DataFrames que se van a reutilizar en los siguientes pasos del proyecto (unión con gasto en fichajes y entrenador), para no depender de recorrer todo el notebook desde cero cada vez:

- `rendimiento_barca.csv`: una fila por temporada, con posición, puntos y estadísticas de rendimiento del Barça.
- `partidos_barca_completo.csv`: los 1,262 partidos individuales del Barça en las 33 temporadas, con goles propios y del rival ya resueltos.

Los datos crudos originales permanecen intactos en `data/raw/`, siguiendo la separación estándar entre datos crudos y datos procesados.

In [33]:
rendimiento_barca.to_csv('../data/processed/rendimiento_barca.csv', index=False)
partidos_barca_completo.to_csv('../data/processed/partidos_barca_completo.csv', index=False)

print('Guardado:', len(rendimiento_barca), 'temporadas')
print('Guardado:', len(partidos_barca_completo), 'partidos')

Guardado: 33 temporadas
Guardado: 1262 partidos


In [34]:
partidos_barca_completo = pd.read_csv('../data/processed/partidos_barca_completo.csv', dtype={'temporada': str})

In [35]:
corte_0203 = '2003-01-31'
corte_1920 = '2020-01-13'

partidos_0203 = partidos_barca_completo[partidos_barca_completo['temporada'] == '0203']
partidos_1920 = partidos_barca_completo[partidos_barca_completo['temporada'] == '1920']

print('02-03 con Van Gaal:', (partidos_0203['Date'] < corte_0203).sum())
print('02-03 con Antić:', (partidos_0203['Date'] >= corte_0203).sum())
print('19-20 con Valverde:', (partidos_1920['Date'] < corte_1920).sum())
print('19-20 con Setién:', (partidos_1920['Date'] >= corte_1920).sum())

02-03 con Van Gaal: 19
02-03 con Antić: 19
19-20 con Valverde: 19
19-20 con Setién: 19


### Tabla de entrenadores por temporada

Se construyó a mano la lista de entrenadores del Barça para las 33 temporadas del rango del proyecto, usando como referencia el listado histórico de Wikipedia y notas de prensa para confirmar fechas exactas de cambios a mitad de temporada.

**Decisión de diseño (enfoque híbrido):** en vez de forzar una sola fila por temporada, se construyó primero una tabla detallada (`entrenadores_detalle`) con un registro por cada tramo de entrenador, incluyendo los casos donde hubo más de uno en la misma temporada. De ahí se deriva una segunda tabla, `entrenadores_temporada`, con una sola fila por temporada (necesaria para el `merge()` posterior con `rendimiento_barca`), agregando dos columnas: `entrenador_principal` y `hubo_cambio_entrenador`. Esto evita perder información real (el cambio de entrenador en sí es una señal analítica) sin complicar la unión final de los datasets.

**Casos con cambio de entrenador a mitad de temporada:** se identificaron seis, cada uno confirmado con fuentes externas (fecha exacta de salida/entrada del entrenador): 1995/96 (Cruyff → Rexach), 2000/01 (Serra Ferrer → Rexach), 2002/03 (Van Gaal → Antić), 2012/13 (Vilanova, con Roura de interino por licencia médica), 2019/20 (Valverde → Setién) y 2021/22 (Koeman → Barjuan interino → Xavi).

**Intento de desempate por número de partidos dirigidos:** en dos de los seis casos (2002/03 y 2019/20), la fecha de cambio cae casi exactamente a la mitad del calendario de esa temporada, así que no era obvio a simple vista quién dirigió más partidos. Se resolvió cruzando la fecha real de cambio contra las fechas de los partidos en `partidos_barca_completo`, contando cuántos le tocaron a cada entrenador. El resultado fue un **empate exacto: 19 partidos para cada uno** en ambos casos (Van Gaal/Antić en 02-03, Valverde/Setién en 19-20).

**Conclusión y criterio de desempate final:** dado que el conteo de partidos no resuelve el empate, se definió una regla explícita y neutral: en caso de empate exacto, `entrenador_principal` es quien **inició** el tramo. Se descartó a propósito cualquier criterio basado en si la temporada terminó bien o mal (por ejemplo, "el responsable de una mala temporada es quien la empezó"), porque eso introduciría el resultado dentro de la definición de la variable explicativa, invalidando cualquier análisis posterior que busque medir el efecto del entrenador sobre el rendimiento. El criterio usado es puramente descriptivo y consistente en todos los casos, no una interpretación narrativa de responsabilidad.

In [36]:
entrenadores_detalle = [
    {'temporada': '9394', 'entrenador': 'Johan Cruyff', 'tipo': 'principal', 'notas': ''},
    {'temporada': '9495', 'entrenador': 'Johan Cruyff', 'tipo': 'principal', 'notas': ''},
    {'temporada': '9596', 'entrenador': 'Johan Cruyff', 'tipo': 'principal', 'notas': 'Destituido 18/05/1996'},
    {'temporada': '9596', 'entrenador': 'Carles Rexach', 'tipo': 'interino', 'notas': 'Últimas 2 jornadas'},
    {'temporada': '9697', 'entrenador': 'Bobby Robson', 'tipo': 'principal', 'notas': ''},
    {'temporada': '9798', 'entrenador': 'Louis van Gaal', 'tipo': 'principal', 'notas': ''},
    {'temporada': '9899', 'entrenador': 'Louis van Gaal', 'tipo': 'principal', 'notas': ''},
    {'temporada': '9900', 'entrenador': 'Louis van Gaal', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0001', 'entrenador': 'Llorenç Serra Ferrer', 'tipo': 'principal', 'notas': 'Destituido 23/04/2001'},
    {'temporada': '0001', 'entrenador': 'Carles Rexach', 'tipo': 'interino', 'notas': 'Resto de temporada'},
    {'temporada': '0102', 'entrenador': 'Carles Rexach', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0203', 'entrenador': 'Louis van Gaal', 'tipo': 'principal', 'notas': 'Salida finales enero 2003. Empate exacto en partidos dirigidos (19-19); queda como principal por convención (quien inició el tramo)'},
    {'temporada': '0203', 'entrenador': 'Radomir Antić', 'tipo': 'interino', 'notas': 'Desde finales enero 2003'},
    {'temporada': '0304', 'entrenador': 'Frank Rijkaard', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0405', 'entrenador': 'Frank Rijkaard', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0506', 'entrenador': 'Frank Rijkaard', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0607', 'entrenador': 'Frank Rijkaard', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0708', 'entrenador': 'Frank Rijkaard', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0809', 'entrenador': 'Pep Guardiola', 'tipo': 'principal', 'notas': ''},
    {'temporada': '0910', 'entrenador': 'Pep Guardiola', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1011', 'entrenador': 'Pep Guardiola', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1112', 'entrenador': 'Pep Guardiola', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1213', 'entrenador': 'Tito Vilanova', 'tipo': 'principal', 'notas': 'Licencia médica dic 2012-mar 2013'},
    {'temporada': '1213', 'entrenador': 'Jordi Roura', 'tipo': 'interino', 'notas': 'Durante licencia médica de Vilanova'},
    {'temporada': '1314', 'entrenador': 'Gerardo Martino', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1415', 'entrenador': 'Luis Enrique', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1516', 'entrenador': 'Luis Enrique', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1617', 'entrenador': 'Luis Enrique', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1718', 'entrenador': 'Ernesto Valverde', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1819', 'entrenador': 'Ernesto Valverde', 'tipo': 'principal', 'notas': ''},
    {'temporada': '1920', 'entrenador': 'Ernesto Valverde', 'tipo': 'principal', 'notas': 'Destituido 13/01/2020. Empate exacto en partidos dirigidos (19-19); queda como principal por convención (quien inició el tramo)'},
    {'temporada': '1920', 'entrenador': 'Quique Setién', 'tipo': 'interino', 'notas': 'Desde 13/01/2020'},
    {'temporada': '2021', 'entrenador': 'Ronald Koeman', 'tipo': 'principal', 'notas': ''},
    {'temporada': '2122', 'entrenador': 'Ronald Koeman', 'tipo': 'principal', 'notas': 'Destituido 27/10/2021'},
    {'temporada': '2122', 'entrenador': 'Sergi Barjuan', 'tipo': 'interino', 'notas': '28/10 al 04/11/2021'},
    {'temporada': '2122', 'entrenador': 'Xavi Hernández', 'tipo': 'principal', 'notas': 'Desde 05/11/2021, dirigió la mayoría de la temporada'},
    {'temporada': '2223', 'entrenador': 'Xavi Hernández', 'tipo': 'principal', 'notas': ''},
    {'temporada': '2324', 'entrenador': 'Xavi Hernández', 'tipo': 'principal', 'notas': ''},
    {'temporada': '2425', 'entrenador': 'Hansi Flick', 'tipo': 'principal', 'notas': ''},
    {'temporada': '2526', 'entrenador': 'Hansi Flick', 'tipo': 'principal', 'notas': 'Temporada en curso'},
]

entrenadores_detalle = pd.DataFrame(entrenadores_detalle)
entrenadores_detalle.to_csv('../data/processed/entrenadores_detalle.csv', index=False)
entrenadores_detalle

,temporada,entrenador,tipo,notas
0,9394,Johan Cruyff,principal,
1,9495,Johan Cruyff,principal,
2,9596,Johan Cruyff,principal,Destituido 18/05/1996
3,9596,Carles Rexach,interino,Últimas 2 jornadas
4,9697,Bobby Robson,principal,
5,9798,Louis van Gaal,principal,
6,9899,Louis van Gaal,principal,
7,9900,Louis van Gaal,principal,
8,0001,Llorenç Serra Ferrer,principal,Destituido 23/04/2001
9,0001,Carles Rexach,interino,Resto de temporada


In [37]:
entrenadores_temporada = (
    entrenadores_detalle[entrenadores_detalle['tipo'] == 'principal']
    .groupby('temporada')
    .first()
    .reset_index()[['temporada', 'entrenador']]
    .rename(columns={'entrenador': 'entrenador_principal'})
)

cambios = entrenadores_detalle.groupby('temporada').size().reset_index(name='num_tramos')
entrenadores_temporada = entrenadores_temporada.merge(cambios, on='temporada')
entrenadores_temporada['hubo_cambio_entrenador'] = entrenadores_temporada['num_tramos'] > 1
entrenadores_temporada = entrenadores_temporada.drop(columns='num_tramos')

entrenadores_temporada.to_csv('../data/processed/entrenadores_temporada.csv', index=False)
entrenadores_temporada

,temporada,entrenador_principal,hubo_cambio_entrenador
0,0001,Llorenç Serra Ferrer,True
1,0102,Carles Rexach,False
2,0203,Louis van Gaal,True
3,0304,Frank Rijkaard,False
4,0405,Frank Rijkaard,False
5,0506,Frank Rijkaard,False
6,0607,Frank Rijkaard,False
7,0708,Frank Rijkaard,False
8,0809,Pep Guardiola,False
9,0910,Pep Guardiola,False
